# Fine-Tune With LoRA and QLoRA Using PEFT

**What you'll learn**
- Add LoRA adapters with PEFT and train them with `Trainer`
- Look inside one LoRA layer: `A`, `B`, rank and alpha
- Save a small adapter, load it back, merge it into the model
- QLoRA: a 4B model in 4-bit, fine-tuned within a free GPU's memory

## Setup
- Same card problem, splits, metric and `Trainer` settings as the previous section

In [ ]:
import os
import torch
import evaluate
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          BitsAndBytesConfig)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, AutoPeftModelForSequenceClassification

NAMES = [
    "card_arrival",
    "card_delivery_estimate",
    "activate_my_card",
    "card_not_working",
    "lost_or_stolen_card",
    "compromised_card",
    "card_payment_not_recognised",
    "declined_card_payment",
    "card_swallowed",
    "contactless_not_working",
]

In [ ]:
banking = load_dataset("mteb/banking77")  # 77 intents, real customer messages
banking

In [ ]:
def is_card_intent(row):
    return row["label_text"] in NAMES  # one of our ten card intents?

cards = banking.filter(is_card_intent)
cards

In [ ]:
def add_label(row):
    return {"label": NAMES.index(row["label_text"])}  # the intent's position in NAMES, 0 to 9

cards = cards.map(add_label)
cards["train"][0]

In [ ]:
split = cards["train"].train_test_split(test_size=200, seed=0)  # 200 training messages kept for validation
split

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tok(batch["text"])  # token ids for a batch of messages

tokenize(split["train"][:2])

In [ ]:
train = split["train"].map(tokenize, batched=True)
val = split["test"].map(tokenize, batched=True)
test = cards["test"].map(tokenize, batched=True)
len(train), len(val), len(test)

In [ ]:
accuracy = evaluate.load("accuracy")
BF16 = torch.cuda.is_bf16_supported()  # can this GPU train in bfloat16?
GB = 1024**3
BF16

In [ ]:
def compute_metrics(p):
    predictions = p.predictions.argmax(-1)  # the top label for each message
    return accuracy.compute(predictions=predictions, references=p.label_ids)

In [ ]:
def trainer(model, lr, epochs=3):
    args = TrainingArguments(
        "runs/lora",
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        bf16=BF16,
        fp16=not BF16,
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="no",
        report_to="none",
    )
    return Trainer(
        model=model,
        args=args,
        train_dataset=train,
        eval_dataset=val,
        processing_class=tok,
        compute_metrics=compute_metrics,
    )

In [ ]:
def new_model(name=MODEL, dtype=torch.float32, quantization_config=None):
    return AutoModelForSequenceClassification.from_pretrained(
        name,
        num_labels=10,  # one output per card intent
        pad_token_id=tok.pad_token_id,  # so the head can find the last real token
        dtype=dtype,
        quantization_config=quantization_config,  # 4-bit loading, used for QLoRA
    )

## Add LoRA Adapters With PEFT
- `r`: the rank of each update; `lora_alpha / r` scales it
- `target_modules="all-linear"`: an adapter beside every linear layer of the body
- `task_type="SEQ_CLS"`: the new `score` head trains in full next to the adapters

In [ ]:
config = LoraConfig(
    task_type="SEQ_CLS",  # the new score head trains in full
    r=16,  # the rank of each update
    lora_alpha=32,  # the update is scaled by lora_alpha / r
    lora_dropout=0.05,
    target_modules="all-linear",  # an adapter beside every linear layer
)
config

In [ ]:
model = new_model()
model = get_peft_model(model, config)  # the same model, now with adapters
model.print_trainable_parameters()

## Inside One LoRA Layer: Frozen Weight Plus A and B
- `base_layer`: the frozen original weight
- `lora_A` maps 896 numbers down to 16, `lora_B` maps them back up

In [ ]:
q = model.base_model.model.model.layers[0].self_attn.q_proj  # the query projection of the first layer
q

In [ ]:
A = q.lora_A["default"].weight
B = q.lora_B["default"].weight
A.shape, B.shape, q.scaling["default"]

In [ ]:
B.abs().max()

- `B` starts at zero: before training, the adapted model gives exactly the original answers

## Train the LoRA Adapters With the Hugging Face Trainer
- LoRA takes a larger learning rate than a full fine-tune: the adapters start from zero

In [ ]:
torch.cuda.reset_peak_memory_stats()
lora = trainer(model, 2e-4)
lora.train()

In [ ]:
scores = lora.evaluate(test)
scores

In [ ]:
torch.cuda.max_memory_allocated() / GB  # the most GPU memory training used

## LoRA's Weight Update Is Low-Rank
- `B @ A` times the scaling: added to the frozen weight at every forward pass

In [ ]:
delta = q.get_delta_weight("default")  # the update LoRA adds to the frozen weight
delta.shape, torch.linalg.matrix_rank(delta.float())

## Save, Load and Merge the LoRA Adapter
- `save_pretrained` on a PEFT model writes the adapter and the new head, not the base model

In [ ]:
OUT = "runs/qwen05-cards-lora"
model.save_pretrained(OUT)  # the adapter and the new head
os.listdir(OUT)

In [ ]:
for name in os.listdir(OUT):
    size = os.path.getsize(f"{OUT}/{name}") / 1024**2  # in MB
    print(name, round(size, 1))

- `AutoPeftModelForSequenceClassification` reads the base model's name from `adapter_config.json`, then adds the adapter

In [ ]:
del lora, model
loaded = AutoPeftModelForSequenceClassification.from_pretrained(OUT, num_labels=10, pad_token_id=tok.pad_token_id)
loaded.base_model.model.model.layers[0].self_attn.q_proj  # the adapter is back

In [ ]:
merged = loaded.merge_and_unload()  # add each update into its weight, drop the adapters
merged.model.layers[0].self_attn.q_proj

In [ ]:
check = trainer(merged, 2e-4)
check.evaluate(test)["eval_accuracy"]

## QLoRA: Load the Base Model in 4-Bit
- A full fine-tune of a 4B model needs about 60 GB; a free T4 has about 15 GB
- `nf4`: 4-bit numbers spaced for normally distributed weights
- Double quantization: the per-block scales are quantized too
- Compute still runs in 16-bit

In [ ]:
del check, loaded, merged
BIG = "Qwen/Qwen3-4B-Instruct-2507"

if BF16:
    compute_dtype = torch.bfloat16
else:
    compute_dtype = torch.float16
compute_dtype

In [ ]:
q4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # 4-bit numbers spaced for normally distributed weights
    bnb_4bit_use_double_quant=True,  # the per-block scales are quantized too
    bnb_4bit_compute_dtype=compute_dtype,  # compute still runs in 16-bit
)
q4

In [ ]:
big = new_model(BIG, dtype="auto", quantization_config=q4)  # the checkpoint's own dtype for what stays unquantized
big.get_memory_footprint() / GB  # GPU memory the 4-bit model takes

In [ ]:
big.num_parameters() * 2 / GB  # the same model in 16-bit: 2 bytes per weight

In [ ]:
big.model.layers[0].self_attn.q_proj, big.score

- Every linear layer of the body is now `Linear4bit`; the new `score` head stays in 16-bit

## QLoRA: Add LoRA Adapters and Train
- `prepare_model_for_kbit_training`: gradient checkpointing on, the weights that are not 4-bit cast to `float32`
- The same `LoraConfig`; the Qwen3 tokenizer is the same as Qwen2.5's, so the tokenized data is reused

In [ ]:
big = prepare_model_for_kbit_training(big)
big.is_gradient_checkpointing

In [ ]:
big = get_peft_model(big, config)  # the same LoRA settings
big.print_trainable_parameters()

In [ ]:
torch.cuda.reset_peak_memory_stats()
qlora = trainer(big, 2e-4)
qlora.train()

In [ ]:
scores = qlora.evaluate(test)
scores

In [ ]:
torch.cuda.max_memory_allocated() / GB

## Lab: Accuracy vs LoRA Rank
- Rank 2 against rank 64 on the 0.5B model: trainable parameters and test accuracy

In [ ]:
del qlora, big
results = []
for r in [2, 64]:
    lab_config = LoraConfig(task_type="SEQ_CLS", r=r, lora_alpha=2 * r, target_modules="all-linear")
    m = get_peft_model(new_model(), lab_config)  # a fresh model with rank-r adapters
    t = trainer(m, 2e-4)
    t.train()
    trainable, total = m.get_nb_trainable_parameters()
    scores = t.evaluate(test)
    results.append({"rank": r, "trainable": trainable, "accuracy": scores["eval_accuracy"]})

results

## Cheat Sheet
| Need | Code |
|---|---|
| LoRA config | `LoraConfig(task_type="SEQ_CLS", r=16, lora_alpha=32, target_modules="all-linear")` |
| Add adapters | `get_peft_model(model, config)` |
| Count | `model.print_trainable_parameters()` |
| Train | the same `Trainer`, learning rate around `2e-4` |
| Save | `model.save_pretrained(path)`: adapter and head only |
| Load | `AutoPeftModelForSequenceClassification.from_pretrained(path)` |
| Merge | `model.merge_and_unload()` |
| 4-bit base | `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)` |
| Before QLoRA | `prepare_model_for_kbit_training(model)` |